In [4]:
from pathlib import Path
import math
import numpy as np

SESSION = Path('../robot/stage_cons_iiwa14/data/demos/BarClean/20260827T151159_249638Z_demo_bar_fixed').resolve()
CSV = SESSION / 'ee_pose.csv'
data = np.genfromtxt(CSV, delimiter=',', names=True)
t = np.asarray(data['time_s'])
position = np.column_stack([data['x_m'], data['y_m'], data['z_m']])
quaternion = np.column_stack([data['qx'], data['qy'], data['qz'], data['qw']])
print(f'samples={len(t)}, duration={t[-1] - t[0]:.3f}s, median_rate={1 / np.median(np.diff(t)):.2f} Hz')

samples=4338, duration=98.509s, median_rate=40.18 Hz


In [5]:
def moving_average(values, window):
    kernel = np.ones(window) / window
    return np.column_stack([np.convolve(values[:, i], kernel, mode='same') for i in range(values.shape[1])])

def runs(mask):
    edges = np.diff(np.r_[False, mask, False].astype(np.int8))
    return list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)))

smooth = moving_average(position, 21)
speed = np.linalg.norm(np.gradient(smooth, t, axis=0), axis=1)
valid = np.arange(len(t)) >= 11
valid &= np.arange(len(t)) < len(t) - 11
for threshold_mm_s in (0.2, 0.5, 1.0, 2.0):
    segments = [(a, b) for a, b in runs(valid & (speed < threshold_mm_s / 1000)) if t[b - 1] - t[a] >= 3.0]
    summary = [(round(t[a], 2), round(t[b - 1], 2), round(t[b - 1] - t[a], 2)) for a, b in segments]
    print(threshold_mm_s, 'mm/s:', summary)

0.2 mm/s: []
0.5 mm/s: []
1.0 mm/s: [(np.float64(0.25), np.float64(3.25), np.float64(3.01)), (np.float64(27.91), np.float64(31.77), np.float64(3.86))]
2.0 mm/s: [(np.float64(0.25), np.float64(3.25), np.float64(3.01)), (np.float64(21.15), np.float64(26.71), np.float64(5.56)), (np.float64(27.13), np.float64(31.84), np.float64(4.71)), (np.float64(74.96), np.float64(80.61), np.float64(5.65)), (np.float64(80.65), np.float64(85.33), np.float64(4.68)), (np.float64(85.37), np.float64(89.67), np.float64(4.3))]


In [6]:
bins = np.arange(0, t[-1] + 5, 5)
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (t >= lo) & (t < hi)
    if not np.any(mask):
        continue
    center = np.median(position[mask], axis=0)
    spread = np.percentile(np.linalg.norm(position[mask] - center, axis=1), 90) * 1000
    median_speed = np.median(speed[mask]) * 1000
    print(f'{lo:5.1f}-{hi:5.1f}s  p={center.round(5)}  p90_spread={spread:5.2f} mm  speed50={median_speed:5.2f} mm/s')

  0.0-  5.0s  p=[ 0.51107 -0.41564  0.5504 ]  p90_spread=212.51 mm  speed50= 0.00 mm/s
  5.0- 10.0s  p=[ 6.7835e-01 -5.7000e-04  2.5607e-01]  p90_spread=137.16 mm  speed50=58.04 mm/s
 10.0- 15.0s  p=[0.65812 0.04589 0.2108 ]  p90_spread=19.73 mm  speed50=13.20 mm/s
 15.0- 20.0s  p=[0.66727 0.05898 0.20597]  p90_spread= 3.68 mm  speed50= 1.84 mm/s
 20.0- 25.0s  p=[0.66657 0.06201 0.20155]  p90_spread= 2.29 mm  speed50= 1.27 mm/s
 25.0- 30.0s  p=[0.66592 0.06747 0.19953]  p90_spread= 2.30 mm  speed50= 0.83 mm/s
 30.0- 35.0s  p=[0.66598 0.06641 0.20039]  p90_spread=39.67 mm  speed50= 4.65 mm/s
 35.0- 40.0s  p=[ 0.65378 -0.01723  0.20461]  p90_spread=41.18 mm  speed50=18.75 mm/s
 40.0- 45.0s  p=[ 0.64144 -0.10645  0.20391]  p90_spread=28.76 mm  speed50=10.93 mm/s
 45.0- 50.0s  p=[ 0.62386 -0.04686  0.27422]  p90_spread=96.98 mm  speed50=38.19 mm/s
 50.0- 55.0s  p=[ 0.64172 -0.04815  0.27628]  p90_spread=61.71 mm  speed50=39.68 mm/s
 55.0- 60.0s  p=[ 0.65379 -0.10654  0.26307]  p90_spread=7

In [7]:
window_a = (t >= 20.0) & (t <= 32.0)
window_b = (t >= 75.0) & (t <= 90.0)
endpoint_a = np.median(position[window_a], axis=0)
endpoint_b = np.median(position[window_b], axis=0)
mad_a = 1.4826 * np.median(np.abs(position[window_a] - endpoint_a), axis=0)
mad_b = 1.4826 * np.median(np.abs(position[window_b] - endpoint_b), axis=0)
center = 0.5 * (endpoint_a + endpoint_b)
delta = endpoint_b - endpoint_a
length_3d = np.linalg.norm(delta)
planar_delta = delta.copy(); planar_delta[2] = 0.0
length_planar = np.linalg.norm(planar_delta)
x_axis = planar_delta / length_planar
z_axis = np.array([0.0, 0.0, 1.0])
y_axis = np.cross(z_axis, x_axis)
rotation = np.column_stack([x_axis, y_axis, z_axis])
yaw = math.atan2(x_axis[1], x_axis[0])
quaternion_bar = np.array([0.0, 0.0, math.sin(yaw / 2), math.cos(yaw / 2)])
print('endpoint A [m]:', endpoint_a, 'robust sigma [mm]:', mad_a * 1000)
print('endpoint B [m]:', endpoint_b, 'robust sigma [mm]:', mad_b * 1000)
print('center [m]:', center)
print(f'length: 3D={length_3d*100:.3f} cm, table-plane={length_planar*100:.3f} cm, vertical mismatch={delta[2]*1000:.3f} mm')
print(f'yaw(first endpoint -> second endpoint)={math.degrees(yaw):.4f} deg')
print('R_base_bar =\n', rotation)
print('quaternion xyzw:', quaternion_bar)
print('orthonormal checks:', rotation.T @ rotation, 'det=', np.linalg.det(rotation))

endpoint A [m]: [0.6661137  0.06581913 0.20015083] robust sigma [mm]: [0.33958828 2.71782823 1.69474062]
endpoint B [m]: [ 0.63350236 -0.24292715  0.20120223] robust sigma [mm]: [1.12400425 3.00456911 0.20806858]
center [m]: [ 0.64980803 -0.08855401  0.20067653]
length: 3D=31.047 cm, table-plane=31.046 cm, vertical mismatch=1.051 mm
yaw(first endpoint -> second endpoint)=-96.0295 deg
R_base_bar =
 [[-0.1050407   0.99446792  0.        ]
 [-0.99446792 -0.1050407   0.        ]
 [ 0.          0.          1.        ]]
quaternion xyzw: [ 0.          0.         -0.74331713  0.6689392 ]
orthonormal checks: [[1.00000000e+00 6.56933674e-18 0.00000000e+00]
 [6.56933674e-18 1.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]] det= 1.0


In [9]:
# Preserve the existing planner's axial sign: endpoint B -> endpoint A.
x_axis_robot = -x_axis
y_axis_robot = np.cross(z_axis, x_axis_robot)
R_robot_bar = np.column_stack([x_axis_robot, y_axis_robot, z_axis])
yaw_robot = math.atan2(x_axis_robot[1], x_axis_robot[0])
q_robot_bar = np.array([0.0, 0.0, math.sin(yaw_robot / 2), math.cos(yaw_robot / 2)])
M_tracker_to_robot = np.array([[0., 0., 1.], [1., 0., 0.], [0., 1., 0.]])
p_bar_robot = np.array([center[0], center[1], 0.10937 + 0.032 / 2])
p_bar_legacy_topic = M_tracker_to_robot.T @ p_bar_robot
R_legacy_topic = M_tracker_to_robot.T @ R_robot_bar

def matrix_to_quaternion(R):
    qw = math.sqrt(max(0.0, 1.0 + np.trace(R))) / 2.0
    if qw > 1e-9:
        return np.array([(R[2,1]-R[1,2])/(4*qw), (R[0,2]-R[2,0])/(4*qw), (R[1,0]-R[0,1])/(4*qw), qw])
    raise ValueError('Calibration rotation unexpectedly has a near-zero quaternion scalar')
q_legacy_topic = matrix_to_quaternion(R_legacy_topic)
print('fixed bar center in iiwa14_link_0:', p_bar_robot)
print(f'fixed yaw in iiwa14_link_0: {math.degrees(yaw_robot):.4f} deg')
print('R_iiwa_bar =\n', R_robot_bar)
print('q_iiwa_bar xyzw:', q_robot_bar)
print('legacy pose_from_iiwa14 xyz:', p_bar_legacy_topic)
print('legacy pose_from_iiwa14 q xyzw:', q_legacy_topic)
print('round-trip position:', M_tracker_to_robot @ p_bar_legacy_topic)
print('round-trip rotation error:', np.max(np.abs(M_tracker_to_robot @ R_legacy_topic - R_robot_bar)))

fixed bar center in iiwa14_link_0: [ 0.64980803 -0.08855401  0.12537   ]
fixed yaw in iiwa14_link_0: 83.9705 deg
R_iiwa_bar =
 [[ 0.1050407  -0.99446792  0.        ]
 [ 0.99446792  0.1050407   0.        ]
 [-0.          0.          1.        ]]
q_iiwa_bar xyzw: [0.         0.         0.6689392  0.74331713]
legacy pose_from_iiwa14 xyz: [-0.08855401  0.12537     0.64980803]
legacy pose_from_iiwa14 q xyzw: [-0.70612816 -0.03718897 -0.03718897  0.70612816]
round-trip position: [ 0.64980803 -0.08855401  0.12537   ]
round-trip rotation error: 0.0


In [8]:
obs_csv = SESSION.parent / '20260827T151934_672672Z_demo_obs_fixed' / 'ee_pose.csv'
obs_data = np.genfromtxt(obs_csv, delimiter=',', names=True)
obs_t = np.asarray(obs_data['time_s'])
obs_position = np.column_stack([obs_data['x_m'], obs_data['y_m'], obs_data['z_m']])
obs_smooth = moving_average(obs_position, 21)
obs_speed = np.linalg.norm(np.gradient(obs_smooth, obs_t, axis=0), axis=1)
obs_bins = np.arange(0, obs_t[-1] + 5, 5)
for lo, hi in zip(obs_bins[:-1], obs_bins[1:]):
    mask = (obs_t >= lo) & (obs_t < hi)
    if not np.any(mask):
        continue
    p = np.median(obs_position[mask], axis=0)
    spread = np.percentile(np.linalg.norm(obs_position[mask] - p, axis=1), 90) * 1000
    print(f'{lo:5.1f}-{hi:5.1f}s  p={p.round(5)}  p90_spread={spread:5.2f} mm  speed50={np.median(obs_speed[mask])*1000:5.2f} mm/s')

  0.0-  5.0s  p=[ 0.66643 -0.17655  0.36431]  p90_spread=49.48 mm  speed50= 5.78 mm/s
  5.0- 10.0s  p=[ 0.5574  -0.01543  0.3613 ]  p90_spread=268.63 mm  speed50=102.28 mm/s
 10.0- 15.0s  p=[0.4978  0.20597 0.36273]  p90_spread=124.18 mm  speed50=84.03 mm/s
 15.0- 20.0s  p=[0.60941 0.11253 0.32635]  p90_spread=106.91 mm  speed50=94.67 mm/s
 20.0- 25.0s  p=[0.72344 0.23292 0.35568]  p90_spread=172.58 mm  speed50=79.89 mm/s
 25.0- 30.0s  p=[0.47267 0.22749 0.51107]  p90_spread=189.45 mm  speed50=131.83 mm/s
 30.0- 35.0s  p=[0.572   0.27736 0.34406]  p90_spread=35.06 mm  speed50=17.74 mm/s
 35.0- 40.0s  p=[0.57182 0.27814 0.34853]  p90_spread= 1.47 mm  speed50= 0.69 mm/s
 40.0- 45.0s  p=[0.57245 0.25737 0.34558]  p90_spread=181.24 mm  speed50=82.92 mm/s
 45.0- 50.0s  p=[0.66978 0.17622 0.32131]  p90_spread=110.59 mm  speed50=75.56 mm/s
 50.0- 55.0s  p=[0.68544 0.11295 0.3201 ]  p90_spread=105.52 mm  speed50=85.38 mm/s
 55.0- 60.0s  p=[0.66405 0.13028 0.32045]  p90_spread=90.96 mm  speed50

In [10]:
obs_hold = (obs_t >= 35.0) & (obs_t <= 40.0)
obs_flange = np.median(obs_position[obs_hold], axis=0)
obs_mad = 1.4826 * np.median(np.abs(obs_position[obs_hold] - obs_flange), axis=0)
obs_center_robot = np.array([obs_flange[0], obs_flange[1], 0.10937 + 0.025])
obs_center_legacy_topic = M_tracker_to_robot.T @ obs_center_robot
print('obstacle flange reference [m]:', obs_flange)
print('obstacle robust sigma [mm]:', obs_mad * 1000)
print('fixed obstacle center in iiwa14_link_0:', obs_center_robot)
print('legacy pose_from_iiwa14 xyz:', obs_center_legacy_topic)

obstacle flange reference [m]: [0.57181573 0.27814383 0.34853313]
obstacle robust sigma [mm]: [0.10363732 0.20726469 1.04420466]
fixed obstacle center in iiwa14_link_0: [0.57181573 0.27814383 0.13437   ]
legacy pose_from_iiwa14 xyz: [0.27814383 0.13437    0.57181573]
